In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install sentence-transformers --quiet
!pip install pandas scikit-learn --quiet


In [ ]:
# Reddit: prefer Kaggle .npz (reddit_data.npz + reddit_graph.npz), else PyG Reddit.
import glob
import logging
import os
import shutil
import numpy as np
import torch
import pandas as pd
from torch_geometric.datasets import Reddit
from torch_geometric.data import Data

REDDIT_ROOT = os.environ.get("REDDIT_DATA_ROOT", "/kaggle/working/reddit_data")
REDDIT_NPZ_SOURCE = os.environ.get(
    "REDDIT_NPZ_SOURCE", "/kaggle/input/datasets/akshatsharma1011/reddit-files"
)
NPZ_DIR = os.path.join(REDDIT_ROOT, "npz_kaggle")
REDDIT_MAX_NODES = int(os.environ.get("REDDIT_MAX_NODES", "250000"))
logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)


def prepare_reddit_npz() -> int:
    if not os.path.isdir(REDDIT_NPZ_SOURCE):
        log.info("Kaggle reddit-files folder not found: %s", REDDIT_NPZ_SOURCE)
        return 0
    os.makedirs(NPZ_DIR, exist_ok=True)
    n = 0
    for p in glob.glob(os.path.join(REDDIT_NPZ_SOURCE, "*.npz")):
        if os.path.isfile(p):
            shutil.copy2(p, os.path.join(NPZ_DIR, os.path.basename(p)))
            n += 1
    log.info("Copied %d .npz file(s) -> %s", n, NPZ_DIR)
    return n


def _pick(arrs, *names, required=True):
    for n in names:
        if n in arrs and arrs[n] is not None:
            return arrs[n]
    if required:
        raise KeyError(f"None of {names} in npz; available: {list(arrs.keys())}")
    return None


def _to_dense_features(feat, num_nodes_hint=None):
    if feat is None:
        return None
    if isinstance(feat, np.ndarray) and feat.dtype == object and feat.size == 1:
        feat = feat.item()
    try:
        from scipy import sparse as sp
        if sp.issparse(feat):
            feat = feat.toarray()
    except Exception:
        pass
    a = np.asarray(feat, dtype=np.float32)
    if a.ndim == 1 and num_nodes_hint and num_nodes_hint > 0:
        if a.size == num_nodes_hint:
            a = a.reshape(-1, 1)
        elif a.size % num_nodes_hint == 0 and a.size != num_nodes_hint:
            a = a.reshape(num_nodes_hint, -1)
    return a


def _build_edge_index(merged) -> np.ndarray:
    ei = _pick(merged, "edge_index", "edges", required=False)
    if ei is not None:
        ei = np.asarray(ei, dtype=np.int64)
        if ei.shape[0] != 2 and ei.size and ei.shape[1] == 2:
            ei = ei.T
        return ei
    if "row" in merged and "col" in merged:
        r = np.asarray(merged["row"], dtype=np.int64).ravel()
        c = np.asarray(merged["col"], dtype=np.int64).ravel()
        if r.size != c.size:
            raise ValueError(f"row {r.size} vs col {c.size}")
        return np.stack([r, c], axis=0)
    raise KeyError(f"No edge_index/edges or row+col; keys: {list(merged.keys())}")


def load_reddit_from_npz():
    gfile = os.path.join(NPZ_DIR, "reddit_graph.npz")
    dfile = os.path.join(NPZ_DIR, "reddit_data.npz")
    if not (os.path.isfile(gfile) and os.path.isfile(dfile)):
        return None
    merged = {}
    for path in (gfile, dfile):
        z = np.load(path, allow_pickle=True)
        for k in z.files:
            merged[k] = z[k]
    y = _pick(merged, "y", "Y", "labels", "label", "y_train", "y_all")
    y = np.asarray(y).reshape(-1).astype(np.int64)
    num_nodes = int(y.shape[0])
    x = _pick(
        merged, "x", "X", "features", "feat", "feature", "Feature", "node_features", required=False
    )
    if x is None:
        raise KeyError(f"No node features; keys: {list(merged.keys())}")
    x = _to_dense_features(x, num_nodes)
    if x.shape[0] != num_nodes:
        if x.ndim == 2 and x.shape[1] == num_nodes:
            x = x.T
        else:
            raise ValueError(f"feature rows {x.shape[0]} != num_nodes {num_nodes}")
    ei = _build_edge_index(merged)
    ei = np.asarray(ei, dtype=np.int64)
    if ei.shape[0] != 2:
        ei = ei.T if ei.shape[1] == 2 else ei
    return Data(
        x=torch.from_numpy(x),
        y=torch.from_numpy(y),
        edge_index=torch.from_numpy(ei).long().contiguous(),
        num_nodes=num_nodes,
    )


def normalize_features(x):
    norm = x.norm(p=1, dim=1, keepdim=True).clamp(min=1.0)
    return x / norm


def subsample_graph(data, max_nodes: int) -> Data:
    n = int(data.num_nodes)
    if n <= max_nodes:
        return data
    dev = data.x.device
    perm = torch.randperm(n, device=dev)[:max_nodes]
    inv = torch.full((n,), -1, dtype=torch.long, device=dev)
    inv[perm] = torch.arange(max_nodes, device=dev, dtype=torch.long)
    e = data.edge_index.to(dev)
    src, dst = e[0], e[1]
    ok = (inv[src] >= 0) & (inv[dst] >= 0)
    new_ei = torch.stack([inv[src[ok]], inv[dst[ok]]], dim=0).contiguous()
    return Data(
        x=data.x[perm],
        y=data.y[perm],
        edge_index=new_ei,
        num_nodes=max_nodes,
    )


prepare_reddit_npz()
raw = load_reddit_from_npz()
if raw is not None:
    log.info("Using Reddit from npz.")
else:
    log.info("Falling back to PyG Reddit (download).")
    raw = Reddit(root=REDDIT_ROOT)[0]
if raw.num_nodes > REDDIT_MAX_NODES:
    log.info("Subsample %d -> %d nodes (REDDIT_MAX_NODES)", raw.num_nodes, REDDIT_MAX_NODES)
    raw = subsample_graph(raw, REDDIT_MAX_NODES)
raw.x = normalize_features(raw.x)
oc = int(raw.y.max().item()) + 1
raw.y = (raw.y * 4 // oc).clamp(0, 3)
nodes = list(range(raw.num_nodes))
Y = np.array([int(raw.y[i].item()) for i in nodes], dtype=int)
edges = pd.DataFrame({
    "id_1": raw.edge_index[0].numpy(),
    "id_2": raw.edge_index[1].numpy(),
})
X = raw.x.numpy()
print("Edges rows:", len(edges), "| X:", X.shape, "| classes:", len(set(Y)))


In [ ]:
import os
REDDIT_ROOT = os.environ.get("REDDIT_DATA_ROOT", "/kaggle/working/reddit_data")
print("Reddit data root:", REDDIT_ROOT)


In [ ]:
import pandas as pd
print(edges.head())
print("Total edge rows:", len(edges))


In [ ]:
import numpy as np
print("Classes:", len(np.unique(Y)))


In [ ]:
print("Feature shape:", X.shape)


In [ ]:
def _top_feature_tokens(row, k=6):
    row = np.asarray(row, dtype=np.float64)
    if row.size == 0:
        return "no features"
    k = min(k, row.size)
    idx = np.argpartition(-row, k - 1)[:k]
    idx = idx[np.argsort(-row[idx])]
    return ", ".join(f"dim_{int(i)}:{float(row[i]):.3f}" for i in idx if row[i] > 0) or "all-zero slice"


documents = []
for node in nodes:
    feats = _top_feature_tokens(X[node])
    doc = (
        f"Node ID: {node}\n"
        f"Reddit post in a co-comment network; strongest normalized input features: {feats}.\n"
        f"Community / class label is not stated in this text.\n"
    )
    documents.append(doc)
print("Sample document:", documents[0])


In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")
text_embeddings = encoder.encode(documents, show_progress_bar=True, batch_size=64)
text_embeddings = np.array(text_embeddings)
print("Text embedding shape:", text_embeddings.shape)


In [ ]:
import torch
node_to_idx = {n: n for n in nodes}
edge_list = []
for _, row in edges.iterrows():
    src_id = int(row["id_1"])
    dst_id = int(row["id_2"])
    if src_id in node_to_idx and dst_id in node_to_idx:
        src, dst = node_to_idx[src_id], node_to_idx[dst_id]
        edge_list.append([src, dst])
        edge_list.append([dst, src])
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print("Edge index shape:", edge_index.shape)


In [ ]:
from torch_geometric.data import Data
import numpy as np

X_combined = np.concatenate([X, text_embeddings], axis=1)
x = torch.tensor(X_combined, dtype=torch.float)
y = torch.tensor(Y, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y)
print(data)


In [ ]:
from sklearn.model_selection import train_test_split
import torch

indices = list(range(len(Y)))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=Y
)
train_mask = torch.zeros(len(Y), dtype=torch.bool)
test_mask = torch.zeros(len(Y), dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True
data.train_mask = train_mask
data.test_mask = test_mask


In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import torch.nn as nn

class GNN_RAG_Model(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        self.bn3 = nn.BatchNorm1d(hidden_channels)
        self.fusion = nn.Linear(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.bn2(self.conv2(h, edge_index)))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.bn3(self.conv3(h, edge_index)))
        h = F.relu(self.fusion(h))
        return self.classifier(h)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GNN_RAG_Model(
    in_channels=data.num_features, hidden_channels=128, num_classes=len(set(Y.tolist()))
).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=50)


In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F

OUTPUT_DIR = "weights"
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_acc = 0.0

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(101):
    loss = train()
    scheduler.step(loss)
    if epoch % 10 == 0:
        model.eval()
        from sklearn.metrics import f1_score, roc_auc_score
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / int(data.test_mask.sum())
        y_te = data.y[data.test_mask].cpu().numpy()
        proba = torch.softmax(out[data.test_mask], dim=1).cpu().numpy()
        p_te = pred[data.test_mask].cpu().numpy()
        f1v = f1_score(y_te, p_te, average="macro", zero_division=0)
        c = proba.shape[1]
        try:
            if c == 2:
                aucv = roc_auc_score(y_te, proba[:, 1])
            else:
                aucv = roc_auc_score(
                    y_te, proba, multi_class="ovr", average="macro", labels=np.arange(c)
                )
        except ValueError:
            aucv = float("nan")
        print(
            f"Epoch {epoch:4d} | Loss: {loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f} "
            f"| Val Acc: {acc:.4f} | Val F1: {f1v:.4f} | Val AUC: {aucv:.4f}"
        )
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pth")


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score

def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())
    y_te = data.y[data.test_mask].cpu().numpy()
    proba = torch.softmax(out[data.test_mask], dim=1).cpu().numpy()
    p_te = pred[data.test_mask].cpu().numpy()
    f1v = f1_score(y_te, p_te, average="macro", zero_division=0)
    c = proba.shape[1]
    try:
        if c == 2:
            aucv = roc_auc_score(y_te, proba[:, 1])
        else:
            aucv = roc_auc_score(
                y_te, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        aucv = float("nan")
    return acc, f1v, aucv

acc, f1v, aucv = test()
print(f"Test Accuracy: {acc:.4f} | F1 (macro): {f1v:.4f} | AUC (macro OVR): {aucv:.4f}")


In [ ]:
# Best checkpoint; output names match training/train_reddit.py
import torch
import torch.nn.functional as F
import numpy as np
from IPython.display import FileLink, display

model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pth", map_location=device))
torch.save(model.state_dict(), f"{OUTPUT_DIR}/model_weights_reddit.pth")
print("Saved model_weights_reddit.pth")

model.eval()
with torch.no_grad():
    x_in, ei = data.x, data.edge_index
    h = F.relu(model.bn1(model.conv1(x_in, ei)))
    h = F.relu(model.bn2(model.conv2(h, ei)))
    h = F.relu(model.bn3(model.conv3(h, ei)))
    embeddings = F.relu(model.fusion(h))

np.save(f"{OUTPUT_DIR}/embeddings_reddit.npy", embeddings.cpu().numpy())
print("Saved embeddings_reddit.npy")
print("Training successfully completed!")

display(FileLink(f"{OUTPUT_DIR}/model_weights_reddit.pth"))
display(FileLink(f"{OUTPUT_DIR}/embeddings_reddit.npy"))


In [ ]:
import numpy as np
import torch
from sklearn.metrics import f1_score, roc_auc_score

def accuracy(pred, y):
    return (pred == y).sum().item() / len(y)

def f1_auc_mask(mask):
    y_ = data.y[mask].cpu().numpy()
    proba = torch.softmax(out[mask], dim=1).cpu().numpy()
    p_ = pred[mask].cpu().numpy()
    f1_ = f1_score(y_, p_, average="macro", zero_division=0)
    c_ = proba.shape[1]
    try:
        if c_ == 2:
            auc_ = roc_auc_score(y_, proba[:, 1])
        else:
            auc_ = roc_auc_score(
                y_, proba, multi_class="ovr", average="macro", labels=np.arange(c_)
            )
    except ValueError:
        auc_ = float("nan")
    return f1_, auc_

model.eval()
out = model(data.x, data.edge_index)
pred = out.argmax(dim=1)
f1_tr, auc_tr = f1_auc_mask(data.train_mask)
f1_te, auc_te = f1_auc_mask(data.test_mask)
print(f"Train Acc: {accuracy(pred[data.train_mask], data.y[data.train_mask]):.4f} | F1: {f1_tr:.4f} | AUC: {auc_tr:.4f}")
print(f"Test Acc: {accuracy(pred[data.test_mask], data.y[data.test_mask]):.4f} | F1: {f1_te:.4f} | AUC: {auc_te:.4f}")
